# Grad-CAM Error Analysis

## Importing Libraries

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import timm
import cv2
import os

print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")

## Building Confident Labels

In [ ]:
LABELS_PATH = '../data/training_solutions_rev1.csv'
IMG_SIZE = 224
BATCH_SIZE = 32

labels_df = pd.read_csv(LABELS_PATH)

def assign_class_confident(row):
    if row['Class1.1'] >= 0.7:
        return 'Smooth'
    elif row['Class1.3'] >= 0.7:
        return None
    elif row['Class4.1'] >= 0.7:
        return 'Spiral'
    elif row['Class1.2'] >= 0.7:
        return 'Irregular'
    else:
        return None

labels_df['label'] = labels_df.apply(assign_class_confident, axis=1)
confident_df = labels_df[labels_df['label'].notna()].copy()

confident_df['filepath'] = confident_df['GalaxyID'].apply(
    lambda gid: os.path.join(IMG_DIR, f"{gid}.jpg")
)

le = LabelEncoder()
confident_df['label_encoded'] = le.fit_transform(confident_df['label'])

print("Classes:", le.classes_)
print(confident_df['label'].value_counts())
print(f"Total: {len(confident_df)}")

## Train/val/test Splitting

In [ ]:
from sklearn.model_selection import train_test_split

filepaths = confident_df['filepath'].values
labels = confident_df['label_encoded'].values

X_train_paths, X_temp_paths, y_train, y_temp = train_test_split(
    filepaths, labels,
    test_size=0.3,
    random_state=42,
    stratify=labels
)

X_val_paths, X_test_paths, y_val, y_test = train_test_split(
    X_temp_paths, y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

print(f"Train: {len(X_train_paths)}")
print(f"Val:   {len(X_val_paths)}")
print(f"Test:  {len(X_test_paths)}")

## Dataset and DataLoader

In [ ]:
class GalaxyDataset(Dataset):
    def __init__(self, filepaths, labels, transform=None):
        self.filepaths = filepaths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        img = Image.open(self.filepaths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return img, label

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_dataset = GalaxyDataset(X_test_paths, y_test, transform=val_transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=2)

print(f"Test batches: {len(test_loader)}")

## Rebuilding and Loading Zoobot Model

In [ ]:
from huggingface_hub import hf_hub_download

# download Zoobot galaxy-pretrained weights
weights_path = hf_hub_download(
    repo_id='mwalmsley/zoobot-encoder-efficientnet_b0',
    filename='pytorch_model.bin'
)

# build encoder
encoder = timm.create_model(
    'efficientnet_b0',
    pretrained=False,
    num_classes=0
)

# load galaxy-pretrained weights
state_dict = torch.load(weights_path, map_location='cpu')
encoder.load_state_dict(state_dict, strict=False)

# build classifier
class ZoobotClassifier(nn.Module):
    def __init__(self, encoder, num_classes=3):
        super(ZoobotClassifier, self).__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
            nn.Linear(1280, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        features = self.encoder(x)
        return self.classifier(features)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ZoobotClassifier(encoder, num_classes=3).to(device)

# load your fine-tuned weights from Drive
checkpoint = torch.load(
    '/content/drive/MyDrive/galaxy-morphology-project/zoobot_finetuned.pt',
    map_location=device
)
model.load_state_dict(checkpoint)
model.eval()
print("Zoobot fine-tuned model loaded successfully.")
print(f"Model on: {device}")

## Getting all Predictions on test set

In [ ]:
all_preds = []
all_labels = []
all_probs = []

model.eval()
with torch.no_grad():
    for images, labels_batch in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        _, predicted = outputs.max(1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels_batch.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

print(f"Total test predictions: {len(all_preds)}")
print(f"Overall accuracy: {(all_preds == all_labels).mean():.4f}")

## Confusion Matrix

In [ ]:
class_names = le.classes_  # ['Irregular', 'Smooth', 'Spiral']

cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, cmap='Blues', colorbar=True)
ax.set_title('Confusion Matrix — Zoobot (90.7% Test Accuracy)', fontsize=14)
plt.tight_layout()
plt.savefig('../reports/figures/confusion_matrix.png',
            dpi=150)
plt.show()
print(cm)

### Confusion Matrix Analysis

The model achieves 90.7% overall accuracy with meaningful variation across
classes. Smooth galaxies are classified most reliably (96.1% recall, 97.6%
precision) — their featureless, round appearance is visually distinctive and
unambiguous. Spiral galaxies have high recall (91.6%) meaning the model finds
most spirals, but lower precision (74.8%) because some Irregular galaxies get
misclassified as Spiral.

The dominant confusion pair is Irregular → Spiral (302 cases, 12.5% of true
Irregulars). This makes scientific sense: irregular and disturbed galaxies
sometimes have partial or distorted spiral-like structure — the same ambiguity
that caused ~40% of the original dataset to be dropped by confidence filtering.
The model is learning the same boundary that human volunteers found hardest to
agree on.

Notably, there are zero Smooth ↔ Spiral confusions in either direction — the
model correctly treats these as visually distinct categories, which is a strong
signal it's learning real morphological features rather than superficial patterns.

## Pre-Class Metrics

In [ ]:
print("=" * 60)
print("Per-Class Classification Report")
print("=" * 60)
report = classification_report(
    all_labels, all_preds,
    target_names=class_names,
    digits=4
)
print(report)

## Grad-CAM Implementation

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()

        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()

        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)

    def generate(self, input_tensor, target_class=None):
        self.model.eval()

        # forward pass
        output = self.model(input_tensor)

        # use predicted class if no target specified
        if target_class is None:
            target_class = output.argmax(dim=1).item()

        # zero gradients
        self.model.zero_grad()

        # backward pass for target class only
        one_hot = torch.zeros_like(output)
        one_hot[0][target_class] = 1
        output.backward(gradient=one_hot)

        # compute Grad-CAM
        # average gradients across spatial dimensions → weight per channel
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)

        # weighted sum of activation maps
        cam = (weights * self.activations).sum(dim=1, keepdim=True)

        # ReLU — only keep positive activations
        cam = torch.relu(cam)

        # normalize to 0-1
        cam = cam - cam.min()
        if cam.max() > 0:
            cam = cam / cam.max()

        # resize to input image size
        cam = cam.squeeze().cpu().numpy()
        cam = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))

        return cam, target_class


# identify the last convolutional layer in EfficientNetB0
# in timm's EfficientNetB0, this is the last block's conv layer
target_layer = model.encoder.blocks[-1][-1].conv_pw

gradcam = GradCAM(model, target_layer)
print("Grad-CAM initialized on last conv layer of EfficientNetB0 encoder.")

## Helper Function to Visualise Grad-CAM

In [ ]:
import matplotlib.cm as mpl_cm

def denormalize(tensor):
    """Convert normalized tensor back to displayable image."""
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img = tensor.cpu() * std + mean
    img = img.clamp(0, 1)
    return img.permute(1, 2, 0).numpy()

def show_gradcam(image_tensor, cam, true_label, pred_label,
                 confidence, class_names, ax_orig, ax_cam):
    """Plot original image and Grad-CAM overlay side by side."""
    img = denormalize(image_tensor)

    # create heatmap — using mpl_cm to avoid conflict with confusion_matrix variable
    heatmap = mpl_cm.jet(cam)[:, :, :3]
    overlay = 0.5 * img + 0.5 * heatmap
    overlay = np.clip(overlay, 0, 1)

    true_name = class_names[true_label]
    pred_name = class_names[pred_label]
    correct = true_label == pred_label

    ax_orig.imshow(img)
    ax_orig.set_title(f"True: {true_name}", fontsize=9)
    ax_orig.axis('off')

    ax_cam.imshow(overlay)
    status = "✓" if correct else "✗"
    color = 'green' if correct else 'red'
    ax_cam.set_title(f"{status} Pred: {pred_name} ({confidence:.1%})",
                     fontsize=9, color=color)
    ax_cam.axis('off')

## Grad-CAM on correctly classified examples

In [ ]:
# get indices of correctly classified examples per class
correct_mask = all_preds == all_labels
n_per_class = 2  # show 2 examples per class

fig, axes = plt.subplots(len(class_names) * n_per_class, 2,
                          figsize=(6, 3 * len(class_names) * n_per_class))

row = 0
for class_idx, class_name in enumerate(class_names):
    # find correctly classified examples for this class
    class_correct = np.where(
        (all_labels == class_idx) & correct_mask
    )[0][:n_per_class]

    for idx in class_correct:
        img_path = X_test_paths[idx]
        true_label = all_labels[idx]
        pred_label = all_preds[idx]
        confidence = all_probs[idx][pred_label]

        # load and preprocess for Grad-CAM
        img = Image.open(img_path).convert('RGB')
        img_tensor = val_transform(img).unsqueeze(0).to(device)

        # generate Grad-CAM
        cam, _ = gradcam.generate(img_tensor, target_class=pred_label)

        show_gradcam(
            img_tensor.squeeze(0), cam,
            true_label, pred_label, confidence,
            class_names, axes[row, 0], axes[row, 1]
        )
        row += 1

plt.suptitle('Grad-CAM: Correctly Classified Examples', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/gradcam_correct.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved correctly classified Grad-CAM examples.")

## Grad-CAM on Misclassified Examples

In [ ]:
# get indices of misclassified examples per class
wrong_mask = all_preds != all_labels

fig, axes = plt.subplots(len(class_names) * n_per_class, 2,
                          figsize=(6, 3 * len(class_names) * n_per_class))

row = 0
for class_idx, class_name in enumerate(class_names):
    # find misclassified examples for this class
    class_wrong = np.where(
        (all_labels == class_idx) & wrong_mask
    )[0][:n_per_class]

    for idx in class_wrong:
        img_path = X_test_paths[idx]
        true_label = all_labels[idx]
        pred_label = all_preds[idx]
        confidence = all_probs[idx][pred_label]

        img = Image.open(img_path).convert('RGB')
        img_tensor = val_transform(img).unsqueeze(0).to(device)

        cam, _ = gradcam.generate(img_tensor, target_class=pred_label)

        show_gradcam(
            img_tensor.squeeze(0), cam,
            true_label, pred_label, confidence,
            class_names, axes[row, 0], axes[row, 1]
        )
        row += 1

plt.suptitle('Grad-CAM: Misclassified Examples', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('/content/Galaxy-Morphology-Classifier/reports/figures/gradcam_misclassified.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved misclassified Grad-CAM examples.")

### Grad-CAM Observations

Grad-CAM heatmaps show what spatial regions of each galaxy image the model
focuses on when making predictions:

- **Smooth galaxies:** heatmaps concentrate on the bright central region and
  overall elliptical shape — consistent with what makes a galaxy look smooth
  and round.
- **Spiral galaxies:** heatmaps highlight the spiral arm regions and disk
  structure — the model is genuinely detecting the structural features that
  define spiral morphology, not just responding to brightness.
- **Irregular galaxies:** heatmaps are more diffuse, often highlighting
  asymmetric or off-center features — consistent with the irregular, disturbed
  nature of these galaxies.

Misclassified examples show the model attending to the wrong features —
for example, an Irregular galaxy misclassified as Spiral shows the heatmap
highlighting a faint curved feature that resembles a spiral arm. This confirms
the confusion is feature-based, not random.

## Error Analysis Summary

In [ ]:
print("=" * 60)
print("ERROR ANALYSIS SUMMARY")
print("=" * 60)

total = len(all_labels)
correct = (all_preds == all_labels).sum()
wrong = total - correct

print(f"\nTotal test examples: {total}")
print(f"Correctly classified: {correct} ({correct/total:.1%})")
print(f"Misclassified: {wrong} ({wrong/total:.1%})")

print("\nConfusion breakdown (row=true, col=predicted):")
for i, true_name in enumerate(class_names):
    for j, pred_name in enumerate(class_names):
        if i != j and cm[i][j] > 0:
            print(f"  {true_name} predicted as {pred_name}: {cm[i][j]} examples "
                  f"({cm[i][j]/cm[i].sum():.1%} of true {true_name})")

print("\nHardest class pairs (most confused):")
off_diag = cm.copy()
np.fill_diagonal(off_diag, 0)
for _ in range(3):
    i, j = np.unravel_index(off_diag.argmax(), off_diag.shape)
    print(f"  {class_names[i]} → {class_names[j]}: {off_diag[i][j]} misclassifications")
    off_diag[i][j] = 0